# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the latest mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Initialize the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
metadata = dataset.metadata  # This is a metadata object, not a dict
print(f"{metadata.name}: {metadata.description}")
print(f"\nVersion: {metadata.version}")
print(f"Published on: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Cite as: {getattr(metadata, 'citeAs', None)}")


## 2. Data Overview
Review available record sets (`@id`), their fields, and columns.

We will list all record sets, fields, and columns and reference them by their `@id` field as required.

In [ ]:
# List all record sets and their fields/columns (@id only, as per instructions)
record_sets = list(dataset.metadata.recordSets)

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # Single field
        fields = [fields]
    print("  Fields and columns:")
    for field in fields:
        field_id = field.get('@id', '<no-id>')
        column_ref = field.get('column', None)
        print(f"    Field @id: {field_id}")
        if column_ref:
            if isinstance(column_ref, dict):
                column_id = column_ref.get('@id', '<no-id>')
                print(f"      Column @id: {column_id}")
            elif isinstance(column_ref, list):
                for col in column_ref:
                    col_id = col.get('@id', '<no-id>')
                    print(f"      Column @id: {col_id}")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

**Reference all entities using their correct `@id`.**
First, identify the main record set for the survey (typically there is a main table).

In [ ]:
# Choose the main record set for extraction. List all available @id to pick one.
all_record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
print('Available record sets:', all_record_set_ids)
# Based on dataset description and convention, the main record set is usually like 'cr:RecordSet' or ends with main
main_record_set_id = all_record_set_ids[0]  # Replace index as needed after inspection
print(f"Selecting record set: {main_record_set_id}\n")

# Extract all data from all record sets into a dictionary of DataFrames
dataframes = {}
for record_set_id in all_record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for {record_set_id}")
    else:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns\n")

# Display the columns for the main record set
if main_record_set_id in dataframes:
    print(f"Main record set columns (@id): {list(dataframes[main_record_set_id].columns)}")
    dataframes[main_record_set_id].head()
else:
    print(f"No dataframe loaded for main record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
We will process the main record set, referencing fields/columns strictly by their `@id`. 

Operations include filtering by a numeric field, normalization, and grouping (e.g., by village or gender).

In [ ]:
# Pick a numeric field for analysis
# View available numeric/score columns, e.g., PHQ-9 total score
cols = dataframes[main_record_set_id].columns
print('Columns in main record set:', cols)

# Example: select PHQ-9 score if available, fallback to another numeric score field by its @id
numeric_field_id = None
for c in cols:
    if 'phq_9_score' in c.lower() or 'phq9' in c.lower():
        numeric_field_id = c
        break
if not numeric_field_id:
    # Try GAD7
    for c in cols:
        if 'gad_7_score' in c.lower() or 'gad7' in c.lower():
            numeric_field_id = c
            break
if not numeric_field_id:
    # Use first numeric-looking column
    for c in cols:
        if dataframes[main_record_set_id][c].apply(lambda x: isinstance(x, (int,float))).any():
            numeric_field_id = c
            break

if numeric_field_id is None:
    raise ValueError("No numeric score field found for demonstration.")

print(f"Using numeric field: {numeric_field_id}\n")

# Filter by threshold (e.g., detecting moderate depression, PHQ-9 > 10)
threshold = 10
filtered_df = dataframes[main_record_set_id].copy()
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the selected field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if present, e.g., gender or village (@id).
group_field_id = None
possible_groups = [c for c in cols if 'gender' in c.lower() or 'village' in c.lower() or 'education' in c.lower()]
if possible_groups:
    group_field_id = possible_groups[0]

if group_field_id:
    print(f"\nGrouping by: {group_field_id}\n")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(grouped_df.head())
else:
    print("No groupable categorical field found.")

## 5. Visualization
Visualize the distribution of the selected numeric score and its relation to a categorical variable if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(dataframes[main_record_set_id][numeric_field_id], bins=20, kde=True, color='teal')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if available)
if group_field_id:
    plt.figure(figsize=(10,4))
    sns.boxplot(data=dataframes[main_record_set_id], x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
This notebook walked through loading, exploring, and analyzing a conformant Croissant-structured dataset using `mlcroissant`.

- **Data loaded using Croissant schema** with all fields referenced by their `@id`.
- **Main record set and numeric fields** were identified and analyzed.
- **Descriptive statistics and visualizations** revealed the distribution and possible group differences in mental health screening scores.

For further analysis, continue to explore other record sets, fields, and association patterns using the strict `@id` referencing as shown here.